In [6]:
# DO NOT CONTAINERISE
# =====
# For debug
! pip list

# Analysis
# -----
# !pip install python-dotenv
# !pip install scikit-learn
# !pip install tensorflow
# !pip install keras


Package                   Version
------------------------- -----------
absl-py                   2.4.0
affine                    2.4.0
aiohappyeyeballs          2.6.2
aiohttp                   3.14.1
aiosignal                 1.4.0
annotated-types           0.7.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
argopy                    1.3.1
asttokens                 3.0.1
astunparse                1.6.3
attrs                     26.1.0
bokeh                     3.9.1
branca                    0.8.2
Brotli                    1.1.0
cached-property           1.5.2
Cartopy                   0.25.0
certifi                   2026.5.20
cffi                      2.0.0
cftime                    1.6.5
charset-normalizer        3.4.7
click                     8.4.1
click-plugins             1.1.1.2
cligj                     0.7.2
cloudpickle               3.1.2
comm                      0.2.3
contextily                1.7.0
contourpy                 1.3.3
cycler               

In [7]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----
# ! pip install -r requirements.txt
# ! pip list
# ! conda list

import os
import sys
from pathlib import Path

from datetime import datetime
from datetime import timedelta

import calendar
from typing import List

import random
from random import randrange

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
import keras
from keras import layers, Model


# base settings
# -----
conf_vlab_name = "ECVs"

param_workflow_name = "workflow name"

# local
# -----
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")
conf_dir_data_local_tmp = os.path.join("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      = "naa-vre-public"
conf_minio_public_bucket_root = f"vl-{conf_vlab_name.lower()}"
conf_minio_public_local_root  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data")

conf_minio_user_bucket        = "naa-vre-user-data"
conf_minio_user_bucket_root   = conf_vlab_name
conf_minio_user_local_root    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   param_workflow_name)
conf_minio_user_local_flog    = os.path.join(conf_minio_user_local_data, "log.md")

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
secret_SERVICE_KEY = ""
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# Input param
# -----
conf_delimiter_tsv = "\t"
conf_delimiter_csv = ","

conf_SERVICE_URL_BEACON_NODE_ACTRIS = "https://beacon-iriscc.maris.nl"
conf_SERVICE_URL_BEACON_NODE_ARGO   = "https://beacon-argo.maris.nl"    # jwt_token=BEACON_TOKEN
conf_SERVICE_URL_BEACON_NODE_CDI    = "https://beacon-cdi.maris.nl"     # jwt_token=BEACON_TOKEN
conf_SERVICE_URL_BEACON_NODE_IAGOS  = "https://beacon-iriscc.maris.nl"
conf_SERVICE_URL_BEACON_NODE_ICOS   = "https://beacon-iriscc.maris.nl"
conf_SERVICE_URL_BEACON_NODE_IRISCC = "https://beacon-iriscc.maris.nl"

# The ML model predicts where a Botritis outbreak is likely to happen
param_fname_metatraining_csv = 'Template-metatraining.csv'
param_fname_observations_csv = 'Template-observations.csv'             # crop disease observations courtesy from AnaEE's BioMA platform
param_fname_weather_csv      = 'Template-weather.csv'
param_fname_keras_model      = "metamodel.keras"

param_data_column_target    = 'Botrite'
param_data_column_train     = [
    'LW',          # 
    'RH',          # relative humidity
    'Rain',        # 
    'Temperature'  # 
]
param_data_column_timestamp = 'RowKey'                                   # datatime
param_data_column_groupby   = "PartitionKey"                             # location

param_data_resample_freq  = "60min"
param_data_seq_len        = 240                                          # resample with a hourly step, 240 steps are 10 days
param_data_response_len   = 24                                           # predict the next 24 hours

param_ml_rand_seed  = 42
param_ml_batch_size = 16
param_ml_epochs     = 12

print("Finish: NaaVRE parameters")
print(f"Workspace public:")
print(f"  Root: {conf_minio_public_local_root}")
print(f"  Code: {conf_minio_public_local_code}")
print(f"  Data: {conf_minio_public_local_data}")

print(f"Workspace user:")
print(f"  Root: {conf_minio_user_local_root}")
print(f"  Code: {conf_minio_user_local_code}")
print(f"  Data: {conf_minio_user_local_data}")
print(f"  Log:  {conf_minio_user_local_flog}")


Finish: NaaVRE parameters
Workspace public:
  Root: /home/jovyan/Cloud Storage/naa-vre-public/vl-ecvs
  Code: /home/jovyan/Cloud Storage/naa-vre-public/vl-ecvs/code
  Data: /home/jovyan/Cloud Storage/naa-vre-public/vl-ecvs/data
Workspace user:
  Root: /home/jovyan/Cloud Storage/naa-vre-user-data/ECVs
  Code: /home/jovyan/Cloud Storage/naa-vre-user-data/ECVs/library
  Data: /home/jovyan/Cloud Storage/naa-vre-user-data/ECVs/workflow name
  Log:  /home/jovyan/Cloud Storage/naa-vre-user-data/ECVs/workflow name/log.md


In [ ]:
# ECVs, Meta learning experiment
# ---
# NaaVRE:
#  cell:
#   inputs:
#   outputs:
#    - file_output_npy: String
# ...

try:
    from dotenv import load_dotenv

    load_dotenv()
except:
    print('No dot-environment')

# prepare folders
# .....
if not os.path.exists(conf_dir_data_local_tmp):
    os.makedirs(conf_dir_data_local_tmp)

# if not os.path.exists(conf_minio_public_local_root):
#     os.makedirs(conf_minio_public_local_root)

if not os.path.exists(conf_minio_user_local_root):
    os.makedirs(conf_minio_user_local_root)

if not os.path.exists(conf_minio_user_local_data):
    os.makedirs(conf_minio_user_local_data)
    
with open(conf_minio_user_local_flog, "w+") as fp_log:
    fp_log.write(f"# {param_workflow_name}\n")

# create log
# .....
print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Start"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)

# input
# -----
# Seed value
seed_value = param_ml_rand_seed

file_metatraining = os.path.join(conf_minio_user_local_data, param_fname_metatraining_csv)
file_observations = os.path.join(conf_minio_user_local_data, param_fname_observations_csv)
file_weather      = os.path.join(conf_minio_user_local_data, param_fname_weather_csv)

column_timestamp = param_data_column_timestamp
column_target    = param_data_column_target
column_groupby   = param_data_column_groupby
column_train     = param_data_column_train

data_resample_freq = param_data_resample_freq
data_seq_len       = int(data_seq_len)
data_response_len  = int(param_data_response_len)

# output
# -----
rtn_data = {
    "X_train": None,
    "Y_train": None
}
fname_output_npy = "temperate-train.npy"
file_output_npy = os.path.join(conf_minio_user_local_data, fname_output_npy)

# func
# -----
def get_cyclic_doy(date, functions=[np.cos, np.sin])->List[float]:
    '''
    returns the day of the year expressed as a list of cyclic functions.
    By default it evaluates sin and cos of the normalized DOY.
    '''
    max_doy = 366 if calendar.isleap(date.year) else 365
    rdoy = (date.timetuple().tm_yday -1)/max_doy
    return [f(rdoy * 2 * np.pi) for f in functions]

def resample_dataframe(data_df, interesting_columns=[], resample_freq='60Min', timestamp_column='timestamp', **kwargs):
    print('\t resampling data')
    data_df[timestamp_column] = pd.to_datetime(data_df[timestamp_column])
    data_df.set_index(timestamp_column, inplace=True)
    return data_df.resample(resample_freq).ffill()[interesting_columns].dropna()

def eng_weather_data(df, column='RH'):
    '''this performs all the feature engineering we need'''
    # scaling relative humidity
    df[column] = df[column]/100

    # evaluating cyclic doy features
    df['cos_doy'] = [get_cyclic_doy(x, [np.cos])[0] for x in df.index]
    df['sin_doy'] = [get_cyclic_doy(x, [np.sin])[0] for x in df.index]
    # returning the engineered data frame and the extra features labels
    return df, ['cos_doy', 'sin_doy']

def build_data_points(df, seq_len, response_seq_len, predictor_cols, response_cols, offset=1, **kwargs):
    """Takes a dataframe and cuts it into sequences of given lenghts, plus
    it returns the following elements as a label. It returns a generator object"""
    begin = 0
    end = seq_len
    while (end + offset + response_seq_len) < len(df):
        yield df[begin:end][predictor_cols].to_numpy(), df[end:(end + response_seq_len)][response_cols].to_numpy()
        begin = begin + offset
        end = end + offset

# start
# -----
# Set seed value
# .....
os.environ['PYTHONHASHSEED'] = str(seed_value)

random.seed(seed_value)
np.random.seed(seed_value)

# chech input the data
# .....
df_meta         = pd.read_csv(file_metatraining)
df_observations = pd.read_csv(file_observations)
df_weather      = pd.read_csv(file_weather)

df_meta.head()
df_observations.head()
df_weather.head()

# positives = pd.to_datetime(df_observations.loc[df_observations[column_target] > 0]['column_timestamp'])
# print(f'Number of Botritis observations: {len(positives)}')
# print('Weather data sample:')

# building training data
# .....
X_train = []
Y_train = []
for _, df_tmp in df_meta.groupby(column_groupby):
    df = resample_dataframe(
        df_tmp,
        interesting_columns=column_train,
        resample_freq=data_resample_freq,
        timestamp_column=column_timestamp)
    
    # feature engineering time
    df, extra_features = eng_weather_data(df)
    column_train_new = column_train + extra_features
    for x, y in build_data_points(df, data_seq_len, data_response_len, column_train_new, column_train):
        X_train.append(x)
        Y_train.append(y)
        #print(f'\tadded {len(x)} data points') 

# converting lists to numpy
X_train = np.array(X_train)
Y_train = np.array(Y_train)
# saving the numpy arrays
np.save('x_train.npy', X_train)
np.save('y_train.npy', Y_train)

# finish
# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")


workflow name


FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/Cloud Storage/naa-vre-user-data/ECVs/workflow name/Template-metatraining.csv'